# Практическое занятие № 15 — аудит достоверности ML-результатов
**ТюмГУ · «Технологии искусственного интеллекта — 1» · 90 минут**

Задача: **аудировать регрессионную модель времени доставки**, а не получать «красивую метрику». Проверяем постановку и признаки, целевую переменную, качество источника, утечки, baseline, ошибки по группам, временную устойчивость и воспроизводимость.

**Важно:** ранее тестовая часть уже могла использоваться в работе № 14; поэтому здесь она **не используется**. Проверяем только `dev = train + validation` во временной кросс-валидации. Это не новый независимый финальный тест. Все числа на `demo_orders_13_14.csv` — демонстрационные.

## 1. Загрузка, очистка и контракт признаков (0–20 мин)
Проверить исходный файл, допустимое время возникновения признаков, правила очистки и количество исключённых записей. Подставьте свой `orders_dirty.csv`; демонстрационный CSV применяется только для отладки.

In [1]:
# ОБЩАЯ ПОДГОТОВКА: выполняется самостоятельно в каждом блокноте.
# В Colab загрузите orders_dirty.csv через боковую панель «Файлы».
# При отсутствии официального файла используйте только demo_orders_13_14.csv
# и ОТМЕТЬТЕ В ОТЧЁТЕ, что результаты демонстрационные.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path('orders_dirty.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('demo_orders_13_14.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Загрузите orders_dirty.csv (учебный источник) либо demo_orders_13_14.csv (только демонстрация).')
source_type = 'ОФИЦИАЛЬНЫЙ/УЧЕБНЫЙ НАБОР' if DATA_PATH.name=='orders_dirty.csv' else 'ДЕМОНСТРАЦИОННЫЕ ДАННЫЕ — НЕ СДАВАТЬ КАК ОСНОВНОЙ ИСТОЧНИК'
print('Источник:', DATA_PATH.name, '|', source_type)
raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
REQUIRED = {'order_id','created_at','distance_km','queue_size','promised_minutes','zone','weather','actual_minutes','late'}
assert REQUIRED.issubset(raw.columns), f'Отсутствуют поля: {sorted(REQUIRED-set(raw.columns))}'
print('Исходный размер:', raw.shape, 'Полных повторов:', int(raw.duplicated().sum()))
print('Типы исходных столбцов:')
print(raw.dtypes.to_string())
if 'courier_load' not in raw.columns:
    raw['courier_load']=np.nan
    print('courier_load отсутствует в источнике; в рабочей копии создан только пустой столбец.')

# Для регрессии actual_minutes — цель; late запрещено использовать как признак.
# Пустой/нечисловой target не подходит для fit и оценки; диапазон 5–240 —
# учебное правило качества, согласованное со словарём данных, а не физический закон.
df=raw.drop_duplicates().copy()
df['created_at']=pd.to_datetime(df['created_at'], errors='coerce')
df['actual_minutes']=pd.to_numeric(df['actual_minutes'], errors='coerce')
missing_target=df['actual_minutes'].isna()
out_of_range=df['actual_minutes'].notna() & ~df['actual_minutes'].between(5,240,inclusive='both')
print('Пропущенная/нечисловая цель:',int(missing_target.sum()),
      'вне учебного диапазона 5–240:',int(out_of_range.sum()))
valid_target=df['actual_minutes'].between(5,240,inclusive='both')
valid_time=df['created_at'].notna()
valid_id=df['order_id'].notna()
print('Цель вне 5–240 или NaN:',int((~valid_target).sum()),
      'невалидных дат:',int((~valid_time).sum()),'пустых ID:',int((~valid_id).sum()))
df=df.loc[valid_target & valid_time & valid_id].copy()
# Если разные строки имеют один ID, студент обязан разобраться с ним до разбиения.
assert df['order_id'].is_unique, 'Конфликтующие order_id: разберите вручную до продолжения.'
df=df.sort_values(['created_at','order_id'],kind='stable').reset_index(drop=True)
assert len(df)>=45, f'Для этой демонстрации нужно хотя бы 45 валидных заказов, найдено {len(df)}'
n=len(df)
n_train=int(n*.6); n_dev=int(n*.8)
train=df.iloc[:n_train].copy(); val=df.iloc[n_train:n_dev].copy(); test=df.iloc[n_dev:].copy()
assert len(train)>0 and len(val)>0 and len(test)>0
assert set(train['order_id']).isdisjoint(val['order_id'])
assert set(train['order_id']).isdisjoint(test['order_id'])
assert set(val['order_id']).isdisjoint(test['order_id'])
assert train['created_at'].max()<=val['created_at'].min()<=test['created_at'].min()
print('После очистки:',n,'train/val/test:',list(map(len,[train,val,test])))
print('Доли частей:',[round(len(part)/n,4) for part in (train,val,test)])
print('Общих order_id между частями: 0; сумма размеров:',len(train)+len(val)+len(test))
print('Временные границы непрерывны по сортировке; разрывы между частями:',
      val.created_at.min()-train.created_at.max(),
      test.created_at.min()-val.created_at.max())
print('Временные диапазоны:')
for name,part in [('train',train),('val',val),('test (не оценивается в работе 15)',test)]:
    print(name, len(part),str(part.created_at.min()),'…',str(part.created_at.max()))

NUM_BASE=['distance_km','queue_size','courier_load','promised_minutes']
CAT=['zone','weather']
DERIVED=['hour_sin','hour_cos','planned_speed_kmh']
NUM=NUM_BASE+DERIVED
INPUT=NUM_BASE+CAT+['created_at']

def make_features(X):
    # Функция НЕ обучается по данным и применяется идентично внутри каждого fold.
    f=X.loc[:,INPUT].copy()
    for k in NUM_BASE:
        f[k]=pd.to_numeric(f[k], errors='coerce')
        f.loc[f[k]<0,k]=np.nan
    f.loc[f['promised_minutes']<=0,'promised_minutes']=np.nan
    for k in CAT:
        s=f[k].astype('string').str.strip()
        s=s.mask(s.str.lower().isin(['','nan','none','null','нет данных','неизвестно']))
        f[k]=s.astype(object).where(s.notna(),np.nan)
    ts=pd.to_datetime(f['created_at'],errors='coerce')
    hour=ts.dt.hour.astype(float)
    f['hour_sin']=np.sin(2*np.pi*hour/24)
    f['hour_cos']=np.cos(2*np.pi*hour/24)
    f['planned_speed_kmh']=60*f['distance_km']/f['promised_minutes']
    f['planned_speed_kmh']=f['planned_speed_kmh'].replace([np.inf,-np.inf],np.nan)
    return f.drop(columns=['created_at'])

num_pipe=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('scale',StandardScaler())])
cat_pipe=Pipeline([('impute',SimpleImputer(strategy='constant',fill_value='неизвестно')),
                   ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
prep=ColumnTransformer([('num',num_pipe,NUM),('cat',cat_pipe,CAT)],remainder='drop')

def build_pipeline(regressor):
    return Pipeline([('derive',FunctionTransformer(make_features,validate=False)),
                     ('prep',clone(prep)),('model',regressor)])

def eval_regression(y_true,y_pred):
    return {'MAE':float(mean_absolute_error(y_true,y_pred)),
            'RMSE':float(np.sqrt(mean_squared_error(y_true,y_pred))),
            'R2':float(r2_score(y_true,y_pred))}
Xtrain=train[INPUT].copy(); ytrain=train['actual_minutes'].copy()
Xval=val[INPUT].copy(); yval=val['actual_minutes'].copy()
# Исторический test хранится отдельно и не используется в работе 15.
assert {'order_id','late','actual_minutes'}.isdisjoint(Xtrain.columns)
print('Входные поля:',INPUT,'Цель: actual_minutes (мин)')
print('Признаков до кодирования:',len(NUM)+len(CAT),'; test-цель и test-метрики не просматривались.')


Источник: orders_dirty.csv | ОФИЦИАЛЬНЫЙ/УЧЕБНЫЙ НАБОР
Исходный размер: (600, 10) Полных повторов: 1
Типы исходных столбцов:
order_id                str
created_at              str
distance_km             str
queue_size          float64
weather                 str
zone                    str
promised_minutes      int64
actual_minutes        int64
late                  int64
vehicle_type            str
courier_load отсутствует в источнике; в рабочей копии создан только пустой столбец.
Пропущенная/нечисловая цель: 0 вне учебного диапазона 5–240: 0
Цель вне 5–240 или NaN: 0 невалидных дат: 0 пустых ID: 0
После очистки: 599 train/val/test: [359, 120, 120]
Доли частей: [0.5993, 0.2003, 0.2003]
Общих order_id между частями: 0; сумма размеров: 599
Временные границы непрерывны по сортировке; разрывы между частями: 0 days 00:51:00 0 days 01:57:00
Временные диапазоны:
train 359 2026-08-25 08:00:00 … 2026-09-28 02:45:00
val 120 2026-09-28 03:36:00 … 2026-10-06 02:42:00
test (не оценивается в рабо

In [2]:
# Аудит источника без подглядывания в test-метрики.
raw_a=pd.concat([train,val],ignore_index=True).copy()  # Аудит меток только в dev; test не читаем.
raw_a['actual_num']=pd.to_numeric(raw_a['actual_minutes'],errors='coerce')
raw_a['promised_num']=pd.to_numeric(raw_a['promised_minutes'],errors='coerce')
raw_a['late_num']=pd.to_numeric(raw_a['late'],errors='coerce')
check=(raw_a['actual_num']-raw_a['promised_num']>15).astype('Int64')
mask=raw_a[['actual_num','promised_num','late_num']].notna().all(axis=1)
conflicts=int((check[mask]!=raw_a.loc[mask,'late_num']).sum())
checks=pd.DataFrame([
 {'Проверка':'Строк в источнике','Значение':len(raw)},
 {'Проверка':'Полные повторы','Значение':int(raw.duplicated().sum())},
 {'Проверка':'Невалидное время доставки в dev','Значение':int((~raw_a['actual_num'].between(5,240)).sum())},
 {'Проверка':'Конфликт late с правилом > 15 мин в dev','Значение':conflicts},
 {'Проверка':'Строк в рабочей таблице','Значение':len(df)},
 {'Проверка':'Строк dev для аудита меток','Значение':len(raw_a)},
 {'Проверка':'Повторных order_id после очистки','Значение':int(df['order_id'].duplicated().sum())},
 {'Проверка':'Нечитаемых дат в сыром наборе','Значение':int(pd.to_datetime(raw['created_at'],errors='coerce').isna().sum())},
 {'Проверка':'Входные признаки доступны до доставки','Значение':set(INPUT).isdisjoint({'late','actual_minutes','order_id'})},
])
print(checks.to_string(index=False))
checks.to_csv('15_source_audit.csv',index=False,encoding='utf-8-sig')
conflict_rows=raw_a.loc[(check!=raw_a['late_num']).fillna(False) & mask,
                        ['order_id','actual_minutes','promised_minutes','late']].copy()
conflict_rows['expected_late']=check.loc[conflict_rows.index].astype(int).to_numpy()
print('Конфликтные метки в dev:')
display(conflict_rows)
conflict_rows.to_csv('15_label_conflicts.csv',index=False,encoding='utf-8-sig')
assert len(conflict_rows)==conflicts
assert not set(INPUT)&{'late','actual_minutes','order_id'}
# Какова политика по конфликтным late? Для регрессии эта метка не используется,
# но проблема качества исходного источника должна быть задокументирована.
print('Конфликтная late исключена из X и не является целью регрессии; заказ не удалён только по этой причине.')


                               Проверка Значение
                      Строк в источнике      600
                         Полные повторы        1
        Невалидное время доставки в dev        0
Конфликт late с правилом > 15 мин в dev        1
                Строк в рабочей таблице      599
             Строк dev для аудита меток      479
       Повторных order_id после очистки        0
          Нечитаемых дат в сыром наборе        0
  Входные признаки доступны до доставки     True
Конфликтные метки в dev:


,order_id,actual_minutes,promised_minutes,late,expected_late
43,ORD-0035,71,68,1,0


Конфликтная late исключена из X и не является целью регрессии; заказ не удалён только по этой причине.


## 2. Временная проверка и базовая модель (20–42 мин)
Сравнить `DummyRegressor(median)` и `Ridge(alpha=10)` на одних и тех же 4 временных фолдах. Считать MAE/RMSE только в проверочных частях. Ни один трансформер не обучается по всей выборке заранее.

In [3]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

dev=pd.concat([train,val],ignore_index=True).sort_values(['created_at','order_id']).reset_index(drop=True)
Xdev=dev[INPUT].copy(); ydev=dev['actual_minutes'].copy()
cv=TimeSeriesSplit(n_splits=4)
assert len(dev)==int(len(df)*.8)==479
assert set(dev['order_id']).isdisjoint(test['order_id'])
models={'Dummy':DummyRegressor(strategy='median'), 'Ridge':Ridge(alpha=10.0)}
fold_rows=[]; oof_rows=[]
for fold,(i_tr,i_va) in enumerate(cv.split(Xdev),start=1):
    tr_min=dev.iloc[i_tr]['created_at'].min()
    tr_max=dev.iloc[i_tr]['created_at'].max()
    va_min=dev.iloc[i_va]['created_at'].min()
    va_max=dev.iloc[i_va]['created_at'].max()
    assert tr_max<va_min
    assert set(dev.iloc[i_tr]['order_id']).isdisjoint(dev.iloc[i_va]['order_id'])
    for model_name,est in models.items():
        pipe=build_pipeline(clone(est))
        pipe.fit(Xdev.iloc[i_tr],ydev.iloc[i_tr])
        pred=pipe.predict(Xdev.iloc[i_va])
        mae=float(mean_absolute_error(ydev.iloc[i_va],pred))
        rmse=float(np.sqrt(mean_squared_error(ydev.iloc[i_va],pred)))
        fold_rows.append({'fold':fold,'model':model_name,'n_train':len(i_tr),
                          'n_validation':len(i_va),
                          'train_min':str(tr_min),'train_max':str(tr_max),
                          'val_min':str(va_min),'val_max':str(va_max),
                          'chronology_ok':bool(tr_max<va_min),
                          'MAE_min':mae,'RMSE_min':rmse})
        for ix,actual,p in zip(i_va,ydev.iloc[i_va],pred):
            oof_rows.append({'index':int(ix),'fold':fold,'model':model_name,
                            'order_id':dev.iloc[ix]['order_id'],
                            'created_at':dev.iloc[ix]['created_at'],
                            'zone':dev.iloc[ix]['zone'],'weather':dev.iloc[ix]['weather'],
                            'actual_minutes':float(actual),'prediction':float(p),
                            'residual':float(actual-p),'AE':float(abs(actual-p))})
folds=pd.DataFrame(fold_rows);oof=pd.DataFrame(oof_rows)
print(folds.round(2).to_string(index=False))
summary=folds.groupby('model')[['MAE_min','RMSE_min']].agg(['mean','std']).round(2)
print('\nСредние и вариативность метрик по фолдам:\n',summary)
folds.to_csv('15_fold_scores.csv',index=False,encoding='utf-8-sig')
oof.to_csv('15_oof_predictions.csv',index=False,encoding='utf-8-sig')
assert set(oof['model'])==set(models) and oof['prediction'].notna().all()
assert len(folds)==8 and folds['chronology_ok'].all() and len(oof)==760


 fold model  n_train  n_validation           train_min           train_max             val_min             val_max  chronology_ok  MAE_min  RMSE_min
    1 Dummy       99            95 2026-08-25 08:00:00 2026-09-10 18:18:00 2026-09-10 20:15:00 2026-09-17 02:24:00           True    17.73     21.35
    1 Ridge       99            95 2026-08-25 08:00:00 2026-09-10 18:18:00 2026-09-10 20:15:00 2026-09-17 02:24:00           True     7.52      9.65
    2 Dummy      194            95 2026-08-25 08:00:00 2026-09-17 02:24:00 2026-09-17 04:21:00 2026-09-23 10:30:00           True    17.40     21.36
    2 Ridge      194            95 2026-08-25 08:00:00 2026-09-17 02:24:00 2026-09-17 04:21:00 2026-09-23 10:30:00           True     8.76     13.13
    3 Dummy      289            95 2026-08-25 08:00:00 2026-09-23 10:30:00 2026-09-23 12:27:00 2026-09-29 18:36:00           True    17.23     20.51
    3 Ridge      289            95 2026-08-25 08:00:00 2026-09-23 10:30:00 2026-09-23 12:27:00 2026-09-29 

## 3. Диагностика по периодам и группам (42–67 мин)
Постройте график MAE по фолдам, распределение остатков и таблицу по зонам. Не сравнивайте группы из 2 и 30 заказов как одинаково надёжные. Остаток `e = факт − прогноз`; положительный означает недооценку времени.

In [4]:
import matplotlib.pyplot as plt
r=oof[oof['model']=='Ridge'].copy()
pivot=folds.pivot(index='fold',columns='model',values='MAE_min')
ax=pivot.plot(kind='bar',figsize=(8,4));ax.set(xlabel='Временной фолд',ylabel='MAE, мин',title='Временная устойчивость моделей');plt.tight_layout();plt.savefig('15_fold_mae.png',dpi=160);plt.show()
fig,ax=plt.subplots(figsize=(7.5,4));ax.scatter(r['prediction'],r['residual'],alpha=.6,s=24)
ax.axhline(0,ls='--');ax.set(xlabel='Прогноз, мин',ylabel='Факт − прогноз, мин',title='Остатки Ridge (только OOF)');fig.tight_layout();fig.savefig('15_residuals.png',dpi=160);plt.show()
by_zone=r.groupby('zone',dropna=False).agg(n=('AE','size'),MAE_min=('AE','mean'),median_AE=('AE','median')).sort_values('n',ascending=False)
by_zone['достаточная_группа_n>=10']=by_zone['n']>=10
print(by_zone.round(2).to_string());by_zone.to_csv('15_groups.csv',encoding='utf-8-sig')
print('Группы с n < 10 — только описательные наблюдения, без устойчивых выводов.')
print('Ridge OOF: средний остаток',round(float(r['residual'].mean()),2),'мин; максимальная абсолютная ошибка',round(float(r['AE'].max()),2),'мин.')
print('Минимальный размер зоны:',int(by_zone['n'].min()),'; групп n<10:',int((~by_zone['достаточная_группа_n>=10']).sum()))


          n  MAE_min  median_AE  достаточная_группа_n>=10
zone                                                     
east    108     8.56       7.70                      True
center  102     8.44       7.18                      True
north    88     8.05       7.07                      True
south    82     7.88       5.57                      True
Группы с n < 10 — только описательные наблюдения, без устойчивых выводов.
Ridge OOF: средний остаток -0.15 мин; максимальная абсолютная ошибка 90.0 мин.
Минимальный размер зоны: 82 ; групп n<10: 0


C:\Users\VladS\git\utmn-master-degree\tii\lb15\tmp\ipykernel_34376\2652156791.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax=pivot.plot(kind='bar',figsize=(8,4));ax.set(xlabel='Временной фолд',ylabel='MAE, мин',title='Временная устойчивость моделей');plt.tight_layout();plt.savefig('15_fold_mae.png',dpi=160);plt.show()
C:\Users\VladS\git\utmn-master-degree\tii\lb15\tmp\ipykernel_34376\2652156791.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.axhline(0,ls='--');ax.set(xlabel='Прогноз, мин',ylabel='Факт − прогноз, мин',title='Остатки Ridge (только OOF)');fig.tight_layout();fig.savefig('15_residuals.png',dpi=160);plt.show()


## 4. Проверка сдвига признаков и условный вердикт (67–82 мин)
Сравните входы в train и validation **не используя test**. Учебные критерии пригодности — заданные для упражнения правила, а не научно подтверждённые пороги промышленной системы. Сформулируйте минимум три риска и меры контроля.

In [5]:
# Проверка качества входов на двух последовательных периодах без чтения test.
shift=[]
for feature in ['distance_km','queue_size','promised_minutes','courier_load']:
    a=pd.to_numeric(train[feature],errors='coerce')
    b=pd.to_numeric(val[feature],errors='coerce')
    shift.append({'feature':feature,'median_train':a.median() if a.notna().any() else np.nan,
                  'median_validation':b.median() if b.notna().any() else np.nan,
                  'q1_train':a.quantile(.25),'q3_train':a.quantile(.75),
                  'q1_validation':b.quantile(.25),'q3_validation':b.quantile(.75),
                  'p95_train':a.quantile(.95),'p95_validation':b.quantile(.95),
                  'missing_train_pct':100*a.isna().mean(),'missing_validation_pct':100*b.isna().mean()})
shift=pd.DataFrame(shift);print(shift.round(2).to_string(index=False))
shift.to_csv('15_feature_shift.csv',index=False,encoding='utf-8-sig')

mae_dummy=float(folds[folds.model=='Dummy'].MAE_min.mean())
mae_ridge=float(folds[folds.model=='Ridge'].MAE_min.mean())
max_fold_ridge=float(folds[folds.model=='Ridge'].MAE_min.max())
# Условный gate: измените его только с объяснением предметной области.
quality_gate={'schema_valid':bool(set(INPUT).isdisjoint({'late','actual_minutes','order_id'})),
 'better_than_baseline':bool(mae_ridge<mae_dummy),
 'all_fold_mae_below_35_min':bool(max_fold_ridge<35.0)}
print('Учебный вердикт:',quality_gate)
print('Учебный технический gate:', 'проходит' if all(quality_gate.values()) else 'не проходит')
verdict='нуждается в доработке'
print('ВЕРДИКТ ДЛЯ ПРИМЕНЕНИЯ:',verdict,'— технический gate не устраняет дефекты источника и ограничения проверки.')
import json
report={'source':DATA_PATH.name,'scope':'development time-CV only; test not used',
        'cv_folds':4,'mae_dummy_min':mae_dummy,'mae_ridge_min':mae_ridge,
        'quality_gate_example':quality_gate,
        'verdict':verdict,'delta_mae_min':mae_dummy-mae_ridge,
        'max_ridge_fold_mae_min':max_fold_ridge,
        'risks':[
          {'risk':'утечка данных','evidence':'late и actual_minutes известны после доставки; preprocessing fit только на train фолда','action':'контракт X и единый Pipeline'},
          {'risk':'временной сдвиг','evidence':'медианы train/validation и MAE по фолдам различаются','action':'мониторинг входов и новая временная проверка'},
          {'risk':'малые группы при детализации','evidence':'сейчас минимум 82 заказа в зоне; при дальнейшем делении n может стать <10','action':'контролировать n и не обобщать малые группы'},
          {'risk':'крупные отдельные ошибки','evidence':'максимальная абсолютная OOF-ошибка Ridge около 90 минут','action':'контроль хвостов ошибок и разбор нетипичных заказов'}],
        'caveats':['synthetic educational data possible','group sizes and time drift matter','not an independent deployment approval']}
Path('15_model_audit.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')

         feature  median_train  median_validation  q1_train  q3_train  q1_validation  q3_validation  p95_train  p95_validation  missing_train_pct  missing_validation_pct
     distance_km           9.0                8.8       4.7      13.0           5.18          13.02       17.2           17.80               0.56                     0.0
      queue_size           9.0               10.0       5.0      14.0           6.00          17.00       21.0           21.05               0.28                     0.0
promised_minutes          51.0               52.0      40.0      61.0          40.00          62.25       73.0           76.10               0.00                     0.0
    courier_load           NaN                NaN       NaN       NaN            NaN            NaN        NaN             NaN             100.00                   100.0
Учебный вердикт: {'schema_valid': True, 'better_than_baseline': True, 'all_fold_mae_below_35_min': True}
Учебный технический gate: проходит
ВЕРДИКТ ДЛ

1401

## 5. Самостоятельная часть и отчёт (82–90 мин)
1. Составьте таблицу «риск → доказательство → возможное искажение → действие» для трёх рисков: leakage, временной сдвиг, малые группы.
2. Сравните качество двух моделей с baseline по средним MAE и разбросу на фолдах.
3. Укажите, почему исторический `test` после работы № 14 нельзя объявлять новым независимым тестом.
4. Дайте краткий вердикт «пригодна / нуждается в доработке / данных недостаточно» со ссылкой на конкретные числа.

**Сдать:** `Группа_Фамилия_З15_Отчет.pdf` (или DOCX) + `15_student.ipynb` + `15_model_audit.json`. Скриншоты **не заменяют** таблицу метрик; блокнот должен выполняться после Restart & Run All.

**Ответы на самостоятельные задания.**

1. Таблица рисков составлена по результатам аудита dev и OOF-прогнозов:

   | Риск | Подтверждение | Возможное искажение | Действие |
   |---|---|---|---|
   | Утечка | `actual_minutes` и `late` становятся известны после доставки; один конфликт `late` обнаружен среди 479 dev-заказов | Завышенная оценка, если включить ответ в X или обучить preprocessing на проверочном фолде | Исключать оба поля из X, проверять контракт, обучать преобразования внутри Pipeline |
   | Временной сдвиг | Верхний квартиль `queue_size` изменился с 14 на train до 17 на validation; p95 `promised_minutes` — с 73 до 76,1 | Ошибка следующего периода может отличаться от средней CV | Мониторить входы и проверять модель на новом временном периоде |
   | Малые группы при детализации | Сейчас минимум зоны — 82 заказа; при делении по зоне и погоде n может стать <10 | Нестабильная групповая MAE и ложный вывод о равномерном качестве | Всегда показывать n и не обобщать малые сегменты |
   | Крупные отдельные ошибки | OOF-прогноз Ridge для ORD-0073 ошибся примерно на 90 минут | Средняя MAE скрывает риск для конкретных заказов | Контролировать хвосты ошибок и разбирать нетипичные входы |

2. При одинаковых четырёх временных фолдах Dummy имеет MAE 17,37 ± 0,27 мин и RMSE 21,05 ± 0,40 мин (среднее ± выборочное SD). Ridge с `alpha=10` имеет MAE 8,26 ± 0,54 мин и RMSE 10,72 ± 1,63 мин. Среднее преимущество Ridge по MAE — 9,10 мин, но её MAE меняется от 7,52 до 8,76 мин. Это выигрыш на dev, а не доказательство промышленной пригодности.
3. Исторический test уже оценивался в работе 14. Повторная проверка той же модели или новых решений на нём не станет независимой: сведения об этом периоде уже доступны исследователю. В lb15 метрики считаются только на dev через OOF-прогнозы; для внешней проверки нужна новая будущая когорта.
4. Вердикт — **нуждается в доработке**. Учебный технический gate пройден (8,26 < 17,37 мин; максимум фолда 8,76 < 35 мин), но один `late` конфликтует с правилом, `courier_load` отсутствует у 100% заказов, максимальная OOF-ошибка около 90 мин, а качество ещё не проверено независимо на новых заказах. Перед применением нужны исправление источника, контроль хвостов и новая временная проверка.
